# PCMCI+: v2a-RSN 220210_F1_run6

This recording-specific notebook applies fixed-lag PCMCI+ to the shared
`n50-e18-r32` v2a analysis profile.

- The selected 18 emitter and 32 receiver traces are identical to c-GC and
  c-GC* for this recording.
- `P_VALUES = [1, 2, 3, 4, 5, 6, 7]`.
- The temporal lag is fixed at one frame (`tau_min=tau_max=1`).
- `p` controls the maximum conditioning complexity via Tigramite's four
  conditioning caps.
- The inferred connectivity estimate is checkpointed after every completed
  `p`.

Primary output:
`outputs/v2a-RSNs/n50-e18-r32/pcmciplus/220210_F1_run6.pkl`


In [ ]:
from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm

PACKAGE_RELATIVE_PATH = Path('src/markovianity_diagnostic')
PROJECT_ROOT = next(
    (
        path
        for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (path / PACKAGE_RELATIVE_PATH).exists()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Could not find '{PACKAGE_RELATIVE_PATH}' from {Path.cwd().resolve()}"
    )

SRC_PATH = PROJECT_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from markovianity_diagnostic.experiments.adapters import analyze_with_pcmciplus  # noqa: E402
from markovianity_diagnostic.experiments.graph_metrics import (  # noqa: E402
    compute_graph_stability_metrics,
)
from markovianity_diagnostic.experiments.v2a_rsn_utils import (  # noqa: E402
    V2A_ANALYSIS_PROFILE,
    binarize_adjacencies,
    build_v2a_stability_summary,
    build_v2a_transition_df,
    completed_p_values,
    get_v2a_selection_metadata,
    load_adjacency_checkpoint,
    load_and_filter_traces,
    save_adjacency_checkpoint,
    setup_recording_paths,
    upsert_method_outputs,
    upsert_summary_json,
    v2a_summary_json_payload,
)

print(f'Project root: {PROJECT_ROOT}')

In [ ]:
RECORDING_NAME = '220210_F1_run6'
METHOD_DIR = 'pcmciplus'
METHOD_LABEL = 'PCMCI+'
ANALYZER = analyze_with_pcmciplus

P_VALUES = [1, 2, 3, 4, 5, 6, 7]
PC_ALPHA = 0.05
SKIP_EXISTING = True
REPLACE_EXISTING_P = False

TIGRAMITE_PARAMS = {
    'algorithm': METHOD_DIR,
    'ci_test': "ParCorr",
    'pc_alpha': PC_ALPHA,
    'tau_min': 1,
    'tau_max': 1,
    'depth_parameter': 'maximum_conditioning_set_size',
    'conditioning_parameters': [
        'max_conds_dim',
        'max_conds_py',
        'max_conds_px',
        'max_conds_px_lagged',
    ],
    'directed_edges_only': True,
    'adjacency_orientation': 'target_by_source',
}

print(f'Recording: {RECORDING_NAME}')
print(f'Method: {METHOD_LABEL}')
print(f'Analysis profile: {V2A_ANALYSIS_PROFILE}')
print(f'Depths: {P_VALUES}')
print(pd.Series(TIGRAMITE_PARAMS, name='value').to_string())

In [ ]:
paths = setup_recording_paths(
    PROJECT_ROOT,
    RECORDING_NAME,
    METHOD_DIR,
    analysis_profile=V2A_ANALYSIS_PROFILE,
)
recording_dir = paths['recording_dir']
recording_output_dir = paths['output_dir']
method_output_dir = paths['method_output_dir']
connectivity_pkl = method_output_dir / f'{RECORDING_NAME}.pkl'
metadata_path = recording_output_dir / 'run_metadata.json'

method_output_dir.mkdir(parents=True, exist_ok=True)
recording_output_dir.mkdir(parents=True, exist_ok=True)

X_neurons_by_frames = load_and_filter_traces(recording_dir, RECORDING_NAME)
trace_selection = get_v2a_selection_metadata(recording_dir, RECORDING_NAME)
if connectivity_pkl.exists():
    if not metadata_path.exists():
        raise FileNotFoundError(
            f'Checkpoint exists without run metadata: {connectivity_pkl}'
        )
    existing_metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
    existing_selection = existing_metadata.get('trace_selection', {})
    if (
        existing_selection.get('selected_cell_indices')
        != trace_selection['selected_cell_indices']
    ):
        raise ValueError(
            'Existing checkpoint trace selection does not match the current '
            'deterministic 50-trace profile'
        )
    if existing_metadata.get('tigramite_params') != TIGRAMITE_PARAMS:
        raise ValueError(
            'Existing checkpoint Tigramite parameters do not match the current '
            'configuration; use a separate output namespace'
        )

X_time_by_neurons = X_neurons_by_frames.T

if X_neurons_by_frames.shape[0] != 50:
    raise ValueError(
        f'Expected 50 selected traces, got {X_neurons_by_frames.shape[0]}'
    )
if trace_selection['selected_cell_indices'] != (
    trace_selection['selected_emitter_indices']
    + trace_selection['selected_receiver_indices']
):
    raise ValueError('Trace-selection metadata has inconsistent cell ordering')

print(f'Trace matrix (neurons x frames): {X_neurons_by_frames.shape}')
print(f'Selection seed: {trace_selection["selection_seed"]}')
print(f'Connectivity checkpoint: {connectivity_pkl.relative_to(PROJECT_ROOT)}')

In [ ]:
adjacencies = load_adjacency_checkpoint(connectivity_pkl)


def write_run_metadata(last_completed_p_value: int | None) -> None:
    payload = {
        'saved_at': datetime.now(timezone.utc).isoformat(),
        'recording': RECORDING_NAME,
        'analysis_profile': V2A_ANALYSIS_PROFILE,
        'trace_selection': trace_selection,
        'method_label': METHOD_LABEL,
        'method_key': METHOD_DIR,
        'trace_shape_neurons_by_frames': list(X_neurons_by_frames.shape),
        'p_values_requested': P_VALUES,
        'p_values_completed': completed_p_values(adjacencies, P_VALUES),
        'last_completed_p_value': last_completed_p_value,
        'tigramite_params': TIGRAMITE_PARAMS,
        'template_notebook': 'notebooks/simulations/pcmciplus/pcmci_plus_singleLag-Markovian.ipynb',
        'connectivity_pickle': str(connectivity_pkl.relative_to(PROJECT_ROOT)),
    }
    metadata_path.write_text(json.dumps(payload, indent=2), encoding='utf-8')


completed_before = completed_p_values(adjacencies, P_VALUES)
print(f'Already completed p_values: {completed_before or "none"}')

for p_value in tqdm(P_VALUES, desc=f'{METHOD_LABEL} p_values', unit='p'):
    if p_value in adjacencies and SKIP_EXISTING:
        tqdm.write(f'Skipping P={p_value}: already checkpointed')
        continue
    if p_value in adjacencies and not REPLACE_EXISTING_P:
        raise ValueError(
            f'P={p_value} already exists in {connectivity_pkl}. '
            'Enable SKIP_EXISTING or REPLACE_EXISTING_P.'
        )

    inferred = ANALYZER(X_time_by_neurons, [p_value], pc_alpha=PC_ALPHA)
    adjacency = (np.asarray(inferred[p_value]) != 0).astype(int)
    if adjacency.shape != (50, 50):
        raise ValueError(
            f'P={p_value} returned shape {adjacency.shape}; expected (50, 50)'
        )
    np.fill_diagonal(adjacency, 0)
    adjacencies[p_value] = adjacency
    save_adjacency_checkpoint(connectivity_pkl, adjacencies)
    write_run_metadata(p_value)
    tqdm.write(
        f'Saved P={p_value} to {connectivity_pkl.relative_to(PROJECT_ROOT)}'
    )

completed_after = completed_p_values(adjacencies, P_VALUES)
missing_p_values = [p for p in P_VALUES if p not in adjacencies]
write_run_metadata(max(completed_after) if completed_after else None)

print(f'Completed p_values: {completed_after}')
if missing_p_values:
    print(f'Missing p_values: {missing_p_values}')
else:
    print('All requested inferred connectivity estimates are complete.')

In [ ]:
analysis_complete = not missing_p_values
if not analysis_complete:
    print('Stability exports skipped until every requested p_value is complete.')
else:
    binary_adjacencies = binarize_adjacencies(
        {p: adjacencies[p] for p in P_VALUES}
    )
    metrics = compute_graph_stability_metrics(binary_adjacencies)
    summary = build_v2a_stability_summary(
        RECORDING_NAME,
        connectivity_pkl,
        binary_adjacencies,
        metrics,
    )
    summary['method'] = METHOD_LABEL
    summary['analysis_profile'] = V2A_ANALYSIS_PROFILE

    transition_df = build_v2a_transition_df(
        RECORDING_NAME,
        connectivity_pkl.name,
        binary_adjacencies,
        metrics,
    )
    transition_df.insert(1, 'method', METHOD_LABEL)
    transition_df.insert(2, 'analysis_profile', V2A_ANALYSIS_PROFILE)

    summary_row = {
        key: value
        for key, value in summary.items()
        if key not in {'edge_counts', 'D_p', 'D_parts'}
    }
    summary_df = pd.DataFrame([summary_row])

    summary_df.to_csv(recording_output_dir / 'summary.csv', index=False)
    transition_df.to_csv(
        recording_output_dir / 'transitions.csv',
        index=False,
    )
    (recording_output_dir / 'summary.json').write_text(
        json.dumps(v2a_summary_json_payload(summary), indent=2),
        encoding='utf-8',
    )

    upsert_method_outputs(summary_df, transition_df, method_output_dir)
    upsert_summary_json(
        v2a_summary_json_payload(summary),
        method_output_dir / 'summary.json',
    )

    print(f'T_obs: {summary["T_obs"]:.6f}')
    print(f'Updated method outputs: {method_output_dir.relative_to(PROJECT_ROOT)}')

In [ ]:
if analysis_complete:
    edge_counts = summary['edge_counts']
    d_parts = summary['D_parts']

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(
        sorted(edge_counts),
        [edge_counts[p] for p in sorted(edge_counts)],
        marker='o',
    )
    axes[0].set(
        xlabel='Maximum conditioning-set size p (fixed lag 1)',
        ylabel='Directed edge count',
        title=f'{METHOD_LABEL}: inferred connectivity size',
    )
    axes[0].set_xticks(P_VALUES)
    axes[0].grid(alpha=0.3)

    transition_depths = sorted(d_parts)
    axes[1].plot(
        transition_depths,
        [d_parts[p]['D_minus'] for p in transition_depths],
        marker='v',
        label='D_minus',
    )
    axes[1].plot(
        transition_depths,
        [d_parts[p]['D_plus'] for p in transition_depths],
        marker='^',
        label='D_plus',
    )
    axes[1].plot(
        transition_depths,
        [summary['D_p'][p] for p in transition_depths],
        marker='o',
        label='D_p',
    )
    axes[1].set(
        xlabel='Maximum conditioning-set size p (fixed lag 1)',
        ylabel='Normalized instability',
        title=f'{METHOD_LABEL}: adjacent-depth instability',
    )
    axes[1].set_xticks(P_VALUES[1:])
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    fig.tight_layout()
    figure_path = recording_output_dir / 'stability_summary.png'
    fig.savefig(figure_path, dpi=200, bbox_inches='tight')
    plt.show()
    print(f'Saved {figure_path.relative_to(PROJECT_ROOT)}')

In [ ]:
if analysis_complete:
    reloaded = load_adjacency_checkpoint(connectivity_pkl)
    assert completed_p_values(reloaded, P_VALUES) == P_VALUES
    assert all(reloaded[p].shape == (50, 50) for p in P_VALUES)

    saved_metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
    assert saved_metadata['analysis_profile'] == V2A_ANALYSIS_PROFILE
    assert (
        saved_metadata['trace_selection']['selected_cell_indices']
        == trace_selection['selected_cell_indices']
    )

    expected_outputs = [
        connectivity_pkl,
        metadata_path,
        recording_output_dir / 'summary.csv',
        recording_output_dir / 'transitions.csv',
        recording_output_dir / 'summary.json',
        recording_output_dir / 'stability_summary.png',
    ]
    missing_outputs = [path for path in expected_outputs if not path.exists()]
    if missing_outputs:
        raise FileNotFoundError(f'Missing outputs: {missing_outputs}')
    print(f'Verified {len(expected_outputs)} outputs for {METHOD_LABEL}.')